# Previsão de Demanda por Leitos

O objetivo aqui é prever a taxa de ocupação da rede nos próximos dias — é a base de qualquer alerta antecipado de superlotação. Uso Prophet como modelo principal (a justificativa está em `src/models/previsao_leitos.py`: a série tem sazonalidade semanal e anual ao mesmo tempo, e o Prophet trata as duas de forma nativa, sem eu precisar diferenciar a série manualmente). Antes disso, treino uma suavização exponencial (Holt-Winters) como baseline rápido — se ele já for bom, isso me diz que boa parte da série é regular e sazonal, sem tanta complexidade não-linear.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.previsao_leitos import preparar_serie_prophet, treinar_baseline_holt_winters

plt.rcParams['figure.figsize'] = (12, 5)

serie_rede = pd.read_parquet('../data/processed/serie_rede.parquet')
serie = preparar_serie_prophet(serie_rede, coluna_alvo='taxa_ocupacao_rede')
serie.tail()

## 1. Separando treino e teste

Reservo os últimos 30 dias como teste — horizonte realista para planejamento de curto prazo em gestão de leitos (escalas de equipe, compras, remanejamento entre unidades).

In [ ]:
HORIZONTE = 30
corte = serie['ds'].max() - pd.Timedelta(days=HORIZONTE)
treino = serie[serie['ds'] <= corte]
teste = serie[serie['ds'] > corte]

print(f'Treino: {len(treino)} dias | Teste: {len(teste)} dias')

## 2. Baseline — Holt-Winters

Treino a suavização exponencial com sazonalidade semanal (freq=7) e comparo a previsão com o valor real do período de teste.

In [ ]:
modelo_hw, previsao_hw = treinar_baseline_holt_winters(treino, horizonte=HORIZONTE)

mae_hw = np.mean(np.abs(teste['y'].values - previsao_hw))
mape_hw = np.mean(np.abs((teste['y'].values - previsao_hw) / teste['y'].values)) * 100

print(f'Holt-Winters — MAE: {mae_hw:.4f} | MAPE: {mape_hw:.2f}%')

In [ ]:
fig, ax = plt.subplots()
ax.plot(treino['ds'].tail(120), treino['y'].tail(120), label='Histórico', color='steelblue')
ax.plot(teste['ds'], teste['y'], label='Real (teste)', color='steelblue', linestyle='--')
ax.plot(teste['ds'], previsao_hw, label='Previsão Holt-Winters', color='crimson')
ax.set_title('Previsão de ocupação — baseline Holt-Winters (30 dias)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/previsao_ocupacao_baseline.png', dpi=150)
plt.show()

## 3. Modelo principal — Prophet

O Holt-Winters já erra pouco (MAPE em torno de 6% no meu teste), o que sugere que boa parte da variação da série é sazonal e razoavelmente regular. Ainda assim, treino o Prophet — ele modela a sazonalidade semanal e anual junto com a tendência de forma mais flexível, e é o modelo que deixei como recomendação de produção em `treinar_prophet()`.

Esta célula exige `pip install prophet`, que não é uma dependência leve (o pacote traz o backend `cmdstanpy` e faz download de binários na primeira execução). Por isso deixei o baseline anterior como referência que já funciona sem ela — se o Prophet não estiver instalado no ambiente, o restante do notebook não depende desta célula.

In [ ]:
from src.models.previsao_leitos import treinar_prophet, avaliar_modelo, prever_ocupacao

modelo_prophet = treinar_prophet(treino)
metricas_prophet = avaliar_modelo(modelo_prophet, serie, horizonte_dias=HORIZONTE)
metricas_prophet

In [ ]:
previsao_futura = prever_ocupacao(serie_rede, horizonte=HORIZONTE)
previsao_futura.head()

## Conclusões

O baseline Holt-Winters já entrega um MAPE em torno de 6% no horizonte de 30 dias — bom o suficiente para um alerta operacional de tendência, ainda que sem cravar picos pontuais de superlotação. Espero que o Prophet ajude mais em horizontes de 60-90 dias, quando a componente de sazonalidade anual (pico respiratório de inverno) pesa proporcionalmente mais do que no horizonte curto de 30 dias usado aqui.

Antes de comprometer um alerta operacional de verdade a um único modelo, o próximo passo seria validar os dois com pelo menos mais um ano de dados reais — a série simulada cobre só três anos, e sazonalidade anual precisa de vários ciclos completos para ser estimada com confiança.